# Enterprise Financial Fraud Risk Intelligence Platform
## Research Notebook 01: Exploratory Data Analysis & Quantitative Transaction Profiling

**Author:** Principal Machine Learning Engineer & Quantitative Risk Architect  
**Benchmark:** PaySim Synthetic Financial Fraud Benchmark (Lopez-Rojas et al.)  
**Dataset Scope:** 6,362,620 transactions across 743 discrete simulation hours (~31 calendar days)  

> **Institutional Regulatory Disclaimer:**  
> PaySim is an agent-based synthetic simulation designed for mobile money fraud research. While it captures authentic transaction topologies, severe class imbalance (~0.129%), and account liquidation patterns, it does not represent actual proprietary customer accounts, internal ledgers, or live payment corridors of JPMorgan Chase or any other commercial bank.

In [ ]:
import os
import duckdb
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# Visual aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams.update({'figure.dpi': 150, 'font.sans-serif': 'Helvetica', 'font.size': 11})

conn = duckdb.connect()
RAW_PARQUET = 'data/raw/paysim.parquet'
print(f"DuckDB version: {duckdb.__version__}")

### 1. Step Range Reconciliation & Structural Audit
The original literature assumes 744 steps ($31 \times 24$ hours). Let us inspect the actual observed step bounds in PaySim.

In [ ]:
audit_df = conn.execute(f'''
SELECT 
    MIN(step) AS min_step,
    MAX(step) AS max_step,
    COUNT(DISTINCT step) AS distinct_steps,
    COUNT(*) AS total_transactions,
    SUM(isFraud) AS total_fraud,
    ROUND(SUM(isFraud) * 100.0 / COUNT(*), 4) AS fraud_prevalence_pct
FROM read_parquet('{RAW_PARQUET}')
''').df()
audit_df

### 2. Transaction Corridor Breakdown & Fraud Isolation
We inspect the distribution across the five payment corridors: `CASH_IN`, `CASH_OUT`, `DEBIT`, `PAYMENT`, and `TRANSFER`.

In [ ]:
corridor_summary = conn.execute(f'''
SELECT 
    type,
    COUNT(*) AS total_volume,
    SUM(isFraud) AS fraud_count,
    ROUND(SUM(isFraud) * 100.0 / COUNT(*), 4) AS fraud_rate_pct,
    ROUND(SUM(amount) / 1e9, 2) AS total_amount_billions,
    ROUND(SUM(CASE WHEN isFraud = 1 THEN amount ELSE 0 END) / 1e9, 2) AS fraud_exposure_billions
FROM read_parquet('{RAW_PARQUET}')
GROUP BY type
ORDER BY total_volume DESC
''').df()
corridor_summary

**Critical Finding:** Fraud is strictly concentrated in `TRANSFER` and `CASH_OUT` corridors. `PAYMENT`, `CASH_IN`, and `DEBIT` contain exactly zero fraudulent transactions.

### 3. Amount Dispersion & Statistical Divergence Tests
Comparing legitimate vs fraudulent transaction amounts on a logarithmic scale.

In [ ]:
# Extract sample for density plotting and pull all fraud records
sample_df = conn.execute(f'''
SELECT amount, isFraud FROM read_parquet('{RAW_PARQUET}') USING SAMPLE 100000 (reservoir, 42)
''').df()

fraud_amounts = conn.execute(f'''
SELECT amount FROM read_parquet('{RAW_PARQUET}') WHERE isFraud = 1
''').df()['amount']

legit_amounts = sample_df[sample_df['isFraud'] == 0]['amount']

# Two-sample Mann-Whitney U test and Kolmogorov-Smirnov test
mwu_res = stats.mannwhitneyu(legit_amounts, fraud_amounts, alternative='two-sided')
ks_res = stats.ks_2samp(legit_amounts, fraud_amounts)

print(f"Legitimate Median Amount:  ${legit_amounts.median():,.2f}")
print(f"Fraudulent Median Amount:  ${fraud_amounts.median():,.2f}")
print(f"Mann-Whitney U statistic:  {mwu_res.statistic:.2e} (p-value: {mwu_res.pvalue:.2e})")
print(f"Kolmogorov-Smirnov stat:   {ks_res.statistic:.4f} (p-value: {ks_res.pvalue:.2e})")

### 4. Account Liquidation: Sender Balance Drain Dynamics
Analyzing the degree to which a transaction liquidates the sender's entire available balance.

In [ ]:
drain_analysis = conn.execute(f'''
SELECT 
    isFraud,
    ROUND(AVG(amount / (oldbalanceOrg + 1e-5)), 4) AS avg_drain_ratio,
    ROUND(MEDIAN(amount / (oldbalanceOrg + 1e-5)), 4) AS median_drain_ratio,
    SUM(CASE WHEN oldbalanceOrg = 0.0 THEN 1 ELSE 0 END) AS orig_zero_bal_count,
    ROUND(SUM(CASE WHEN oldbalanceOrg = 0.0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS orig_zero_bal_pct
FROM read_parquet('{RAW_PARQUET}')
WHERE type IN ('TRANSFER', 'CASH_OUT')
GROUP BY isFraud
''').df()
drain_analysis

### 5. Diurnal Seasonality vs. Fraud Attack Rates
Legitimate customer transactions obey a 24-hour diurnal cycle, whereas automated fraud operates constantly.

In [ ]:
diurnal_df = conn.execute(f'''
SELECT 
    (step % 24) AS hour_of_day,
    COUNT(*) AS total_tx,
    SUM(isFraud) AS fraud_tx,
    ROUND(SUM(isFraud) * 100.0 / COUNT(*), 4) AS fraud_rate_pct
FROM read_parquet('{RAW_PARQUET}')
GROUP BY (step % 24)
ORDER BY hour_of_day
''').df()
diurnal_df.head(10)

### 6. Fraud Value Concentration (Pareto Loss Curve)
Measuring what percentage of monetary loss is driven by the top deciles of fraud transactions.

In [ ]:
sorted_fraud = fraud_amounts.sort_values(ascending=False).values
cum_value = np.cumsum(sorted_fraud) / np.sum(sorted_fraud) * 100.0
pct_tx = np.linspace(0, 100, len(sorted_fraud))

idx_20 = int(0.20 * len(sorted_fraud))
idx_50 = int(0.50 * len(sorted_fraud))

print(f"Total Fraud Exposure: ${np.sum(sorted_fraud):,.2f}")
print(f"Top 20% Frauds account for: {cum_value[idx_20]:.2f}% of dollar losses")
print(f"Top 50% Frauds account for: {cum_value[idx_50]:.2f}% of dollar losses")

### 7. Feature Correlation & Risk Linkages
Examining linear dependencies among pre-transaction balances, amounts, and targets.

In [ ]:
corr_df = conn.execute(f'''
SELECT 
    amount,
    oldbalanceOrg,
    oldbalanceDest,
    amount / (oldbalanceOrg + 1e-5) AS drain_ratio,
    (oldbalanceOrg - amount) AS balance_coverage,
    isFraud
FROM read_parquet('{RAW_PARQUET}')
USING SAMPLE 50000 (reservoir, 42)
''').df()
corr_df.corr().round(3)

### 8. Quantitative Takeaways & Feature Engineering Roadmap
1. **Corridor Filtering:** Exclude or downweight non-fraud corridors or use corridor one-hot flags.
2. **Account Liquidation Ratios:** `balance_orig_drain_ratio` and `orig_balance_coverage` are paramount indicators of account takeover.
3. **Point-in-Time Rolling Velocity:** Adversaries execute rapid bursts; historical windows ($1	ext{h}, 6	ext{h}, 24	ext{h}, 72	ext{h}$) must capture velocity without future leakage.
4. **Value-Focused Evaluation:** Model thresholds should be calibrated not merely on transaction count recall, but on **Fraud Value Captured (FVC)**.